# Agent Reliability Lab — Colab GPU pilot

Status: **not executed on hosted GPUs at authoring**. This is an eight-case small-model JSON action-schema pilot, not the multi-step recovery evaluation. Read every cell. Free GPU allocation/Internet access is not guaranteed. No token or ngrok is required. Select a GPU and enable notebook Internet access if available. Model: Qwen/Qwen3-0.6B, Apache-2.0; exact revision is pinned in scripts/gpu_pilot.py. Review its model card before download.


In [ ]:
import subprocess, sys
# Explicit hosted-runtime install, not a laptop install. Inspect before running.
# Uses the runtime's existing PyTorch; its exact version is recorded in the receipt.
subprocess.run([sys.executable, "-m", "pip", "install", "transformers==4.57.1", "safetensors==0.6.2"], check=True)


In [ ]:
import os, subprocess, tempfile
from pathlib import Path
REPO = "https://github.com/sourodeepkundu2025-hub/agent-reliability-lab.git"
# Replace with a reviewed immutable release commit. Never execute a moving branch.
REVIEWED_COMMIT = "SET_REVIEWED_COMMIT_BEFORE_RUNNING"
if len(REVIEWED_COMMIT) != 40 or any(c not in "0123456789abcdef" for c in REVIEWED_COMMIT):
    raise RuntimeError("Set the reviewed 40-character Git commit before executing downloaded code.")
base = Path("/content" if Path("/content").exists() else "/kaggle/working")
# Always use a new clone: preserve earlier notebook edits and interrupted evidence.
work = Path(tempfile.mkdtemp(prefix="agent-reliability-", dir=base))
subprocess.run(["git", "clone", "--no-checkout", REPO, str(work)], check=True)
origin = subprocess.check_output(["git", "-C", str(work), "remote", "get-url", "origin"], text=True).strip()
if origin != REPO:
    raise RuntimeError("Unexpected source repository.")
subprocess.run(["git", "-C", str(work), "fetch", "origin", REVIEWED_COMMIT], check=True)
subprocess.run(["git", "-C", str(work), "switch", "--detach", REVIEWED_COMMIT], check=True)
actual = subprocess.check_output(["git", "-C", str(work), "rev-parse", "HEAD"], text=True).strip()
dirty = subprocess.check_output(["git", "-C", str(work), "status", "--porcelain", "--untracked-files=all"], text=True).strip()
if actual != REVIEWED_COMMIT or dirty:
    raise RuntimeError("Reviewed source checkout is not clean; preserve it and start a new clone.")
os.chdir(work)
print("Reviewed source checkout ready:", REVIEWED_COMMIT, work)


In [ ]:
# Test owned code before loading a model. This does not execute an LLM.
import subprocess, sys
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"], check=True)
# Pilot explicitly uses CUDA:0; a missing GPU fails rather than falling back.
# A unique result folder prevents merging different sessions/hardware.
subprocess.run([sys.executable, "scripts/gpu_pilot.py"], check=True)


## Export and stop

The final cell prints the exact evidence directory under artifacts/gpu-pilot/. Download its receipt.json and traces.jsonl, including incomplete or invalid results. Kaggle users should save notebook outputs; Colab users should explicitly download/persist evidence before runtime expiry. This pilot never resumes or overwrites a run: a restart creates a separate run directory and is not an independent statistical seed. Do not publish account secrets or private datasets. This is not a checkpoint-policy improvement study. A later experiment must connect real model decisions to the closed-world environment with matched budgets and disjoint evaluation content. No tunnel or keep-alive loop is started.
